##### AUTOGRAD Examples

In [1]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv")

In [3]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [4]:
df.drop(columns=['id', 'Unnamed: 32'], inplace = True)

In [5]:
print(df.shape)
df.head()

(569, 31)


,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [6]:
xtrain, xtest, ytrain, ytest = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

##### Scaling

In [7]:
scalar = StandardScaler()
xtrain = scalar.fit_transform(xtrain)
xtest = scalar.transform(xtest)

##### Label Encoding

In [8]:
encoder = LabelEncoder()
ytrain = encoder.fit_transform(ytrain)
ytest = encoder.transform(ytest)

#### Numpy Array to Pytorch Tensors

In [9]:
xtrain = torch.from_numpy(xtrain)
xtest = torch.from_numpy(xtest)
ytrain = torch.from_numpy(ytrain)
ytest = torch.from_numpy(ytest)

In [10]:
print(f"X_train shape: {xtrain.shape}")
print(f"X_test shape: {xtest.shape}")
print(f"Y_train shape: {ytrain.shape}")
print(f"Y_test shape: {ytest.shape}")

X_train shape: torch.Size([455, 30])
X_test shape: torch.Size([114, 30])
Y_train shape: torch.Size([455])
Y_test shape: torch.Size([114])


#### Defining the Neural Netwwork

In [11]:
class MySimpleNeuralNetwork():
    def __init__(self, X: torch.Tensor):
        self.weights = torch.rand(size=(X.shape[1], ), dtype=torch.float64, requires_grad=True)
        self.bias = torch.zeros(size=(1, ), dtype=torch.float64, requires_grad=True)

    def forward(self, X: torch.Tensor):
        z = X @ self.weights + self.bias
        y_pred = torch.sigmoid(z)
        return y_pred

    def loss_func(self, y_pred, y):
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)
        loss = -(ytrain * torch.log(y_pred) + (1 - ytrain) * torch.log(1 - y_pred)).mean()
        return loss

#### Important Parameters

In [12]:
learning_rate = 0.1
epochs = 25

#### Training Pipeline

In [13]:
# create model
model = MySimpleNeuralNetwork(X=xtrain)

for epoch in range(epochs):

    y_pred = model.forward(xtrain) # forward pass

    loss = model.loss_func(y_pred=y_pred, y=ytrain) # compute loss

    loss.backward() # backward pass

    #parameter update
    with torch.no_grad():
        if model.weights.grad is not None:
            model.weights -= learning_rate * model.weights.grad
        if model.bias.grad is not None:
            model.bias -= learning_rate * model.bias.grad

    # reset gradients to zero
    if model.weights.grad is not None:
        model.weights.grad.zero_()
    if model.bias.grad is not None:
        model.bias.grad.zero_()

    # print loss in each epoch
    print(f"Epoch = {epoch + 1}, Loss = {loss}")

Epoch = 1, Loss = 0.3647807580315593
Epoch = 2, Loss = 0.3557749802543403
Epoch = 3, Loss = 0.34709585072362936
Epoch = 4, Loss = 0.33873223579133643
Epoch = 5, Loss = 0.33067188617413085
Epoch = 6, Loss = 0.32290197872068055
Epoch = 7, Loss = 0.3154096959245881
Epoch = 8, Loss = 0.3081827224844736
Epoch = 9, Loss = 0.3012095861989057
Epoch = 10, Loss = 0.29376213695653514
Epoch = 11, Loss = 0.2853859073143407
Epoch = 12, Loss = 0.27729205418410074
Epoch = 13, Loss = 0.26947209159966123
Epoch = 14, Loss = 0.26191807149655477
Epoch = 15, Loss = 0.2546222272580365
Epoch = 16, Loss = 0.24757669268349444
Epoch = 17, Loss = 0.24077332787531605
Epoch = 18, Loss = 0.2342036604496464
Epoch = 19, Loss = 0.22785892827858154
Epoch = 20, Loss = 0.22173019850538517
Epoch = 21, Loss = 0.2158085309612992
Epoch = 22, Loss = 0.21008515546486686
Epoch = 23, Loss = 0.20455163826004466
Epoch = 24, Loss = 0.19920002062589107
Epoch = 25, Loss = 0.19402292104782135


In [14]:
print(model.weights)
print("\n")
print(model.bias)

tensor([ 0.3002,  0.1417,  0.7599,  1.0040,  0.8342,  0.2009, -0.0564,  0.6091,
         0.4497,  0.2085,  0.5123,  0.2990,  0.4502,  0.9787, -0.0442,  0.0413,
         0.1569,  0.5108,  0.8115,  0.0409,  0.6764,  0.9336,  0.1826,  0.8581,
         0.8923,  0.4489,  0.2214,  0.5035,  0.1448,  0.3059],
       dtype=torch.float64, requires_grad=True)


tensor([-0.0545], dtype=torch.float64, requires_grad=True)


#### Model Evaluation

In [16]:
with torch.no_grad():
    y_pred = model.forward(xtest)
    y_pred = (y_pred > 0.5).float()
    accuracy = (y_pred == ytest).float().mean()

print(f"Accuracy = {accuracy.item()}")

Accuracy = 0.8859649300575256
